# Homework 2: Recipe Bot Error Analysis

This notebook walks through the complete error analysis process for a Recipe Bot. We'll identify failure modes, generate test queries, and analyze bot responses to build a taxonomy of errors.

**Note:** This uses the pre-existing queries and bot responses in `results_20250518_215844.csv` as our data source.

For a recording of the homework walkthrough please see: https://youtu.be/h9oAAAYnGx4

In [4]:
# !pip install claudette

In [5]:
from textwrap import dedent

In [6]:
import pandas as pd

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()  # Loads .env into environment
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
# print(api_key)

In [8]:
# import os
# "ANTHROPIC_API_KEY" not in os.environ

In [9]:
from claudette import models, Client
model = models[1]
c = Client(model)
model

'claude-sonnet-4-20250514'

## Part 1: Define Dimensions & Generate Initial Queries

### Identify Key Dimensions


# Core parameters (with example values)

1. **dessert_type** — brownies, cookies, cupcakes, cheesecake, bars, mousse, parfait, pie, truffles, milkshake, ice-cream, blondies, donuts, snack mix, lava cake, pudding, tart, soufflé, fudge, banana bread, s'mores, ice cream sandwich, cake pops, whoopie pies, cereal treats

2. **key_ingredient** — peanut butter cups, mini cups, Reese’s Pieces, peanut butter (creamy/crunchy), cocoa powder, dark chocolate, milk chocolate, white chocolate, pretzels, bananas, oats, graham crackers, caramel, marshmallows, toffee bits, coffee, espresso powder, coconut flakes, peanut brittle, rice cereal, cornflakes, almonds, pecans, hazelnuts, walnuts, strawberries, raspberries, honey, maple syrup, sea salt

3. **event** — birthday, game day, Halloween, Valentine’s, bake sale, kid party, potluck, movie night, holiday (Christmas/Thanksgiving), summer BBQ, New Year’s Eve, Easter, baby shower, bridal shower, graduation party, office party, picnic, family reunion, Super Bowl, fall festival, winter holiday market

4. **flavor_profile** — classic PB-chocolate, salty-sweet, extra chocolatey, nutty, caramel-PB, PB-banana, mocha-PB, spicy-PB (chipotle), maple-PB, PB-cookie dough, chocolate-hazelnut, coconut-PB, PB-toffee crunch, PB-strawberry, PB-pretzel, PB-s’mores, PB-caramel swirl, dark chocolate-orange, PB-mint, PB-cinnamon roll

5. **brand_connection_mode** — literal (uses Reese’s candy), flavor match (PB + chocolate), visual palette (orange/brown/cream), playful nod (branding/stencils, wrappers)

6. **dietary_style** — standard, gluten-free, dairy-free, vegan, high-protein, lower-sugar, peanut-free “thematic” (sunflower butter + color palette)

7. **time_budget** — 10-min no-bake, 30-min quick, 1-hr standard, make-ahead/overnight

8. **allergen_notes** — contains peanuts, contains dairy, contains gluten, peanut-free alt, dairy-free alt, gluten-free alt


### Generate Unique Combinations

In [15]:
prompt = dedent('''
I am designing a Reese’s Dessert Recipe Chat Bot and want to test it with a diverse set of user scenarios. \
Please generate 50 unique combinations (tuples) using the following key dimensions and their possible values:

* **Dessert Type:** brownies, cookies, cupcakes, cheesecake, bars, mousse, parfait, pie, truffles, milkshake, ice-cream, \
blondies, donuts, snack mix, lava cake, pudding, tart, soufflé, fudge, banana bread, s'mores, ice cream sandwich, cake pops, whoopie pies, cereal treats
* **Key Ingredient:** peanut butter cups, mini cups, Reese’s Pieces, peanut butter (creamy/crunchy), cocoa powder, dark chocolate, milk chocolate, white chocolate, pretzels, bananas, oats, graham crackers, caramel, marshmallows, toffee bits, coffee, espresso powder, coconut flakes, peanut brittle, rice cereal, cornflakes, almonds, pecans, hazelnuts, walnuts, strawberries, raspberries, honey, maple syrup, sea salt
* **Event:** birthday, game day, Halloween, Valentine’s, bake sale, kid party, potluck, movie night, holiday (Christmas/Thanksgiving), summer BBQ, New Year’s Eve, Easter, baby shower, bridal shower, graduation party, office party, picnic, family reunion, Super Bowl, fall festival, winter holiday market
* **Flavor Profile:** classic PB-chocolate, salty-sweet, extra chocolatey, nutty, caramel-PB, PB-banana, mocha-PB, spicy-PB (chipotle), maple-PB, PB-cookie dough, chocolate-hazelnut, coconut-PB, PB-toffee crunch, PB-strawberry, PB-pretzel, PB-s’mores, PB-caramel swirl, dark chocolate-orange, PB-mint, PB-cinnamon roll
* **Dietary Style:** standard diet, gluten-free, dairy-free, vegan, high-protein, lower-sugar, peanut-free “thematic” (sunflower butter + color palette)
* **Time Budget:** 10-min no-bake, 30-min quick, 1-hr, make-ahead/overnight

Each combination should select appropriate values from each dimension. In most cases do NOT use all parameters instead normally \
only 3-5 parameters should be used for any single combination. \
Make each value combination natural and ensure they are reaslistic and make sense together. 

Present the results as a Python list of tuples, where each tuple contains dimension values.\
The number of dimensions should vary from 3-5 and be evenly distrubted accross this range of dimensions. 
For example, if 90 dimensions are genereted than you should produce 90/3 = 30 of each dimension length.

[(Dessert Type, Event, Flavor Profile, Dietary Style), (Dessert Type, Flavor Profile, Dietary Style), (Dietary Style, Time_budget, Event, Flavor Profile, Dietary Style) ....]

Ensure that the combinations are varied in length, balanced, and realistic, reflecting a mix of everyday treats, \
holiday specials, and creative Reese’s-themed desserts.

Return only text that can be turned into a python list without any alteration.
''')


In [16]:
# print(prompt)

In [17]:
res = c(prompt)

In [18]:
# Convert to a list of tuples
import ast
tup_list = res.content[0].text
tup_list = ast.literal_eval(tup_list)
print(len(tup_list))

50


In [19]:
# tuple dimensions
from collections import Counter
Counter([len(t) for t in tup_list])

Counter({5: 22, 4: 17, 3: 11})

In [20]:
tup_list

[('brownies', 'birthday', 'classic PB-chocolate', 'standard diet'),
 ('cookies', 'bake sale', 'salty-sweet', 'gluten-free', '30-min quick'),
 ('cupcakes', 'kid party', 'PB-banana'),
 ('cheesecake', "Valentine's", 'PB-strawberry', 'make-ahead/overnight'),
 ('bars', 'game day', 'PB-pretzel', 'standard diet', '1-hr'),
 ('mousse', 'PB-toffee crunch', 'dairy-free'),
 ('parfait', 'summer BBQ', 'PB-caramel swirl', '10-min no-bake'),
 ('pie', 'Thanksgiving', 'maple-PB', 'standard diet', 'make-ahead/overnight'),
 ('truffles', "Valentine's", 'dark chocolate-orange', '30-min quick'),
 ('milkshake', 'movie night', 'classic PB-chocolate'),
 ('ice-cream',
  'summer BBQ',
  'PB-cookie dough',
  'standard diet',
  'make-ahead/overnight'),
 ('blondies', 'office party', 'caramel-PB', 'gluten-free'),
 ('donuts', 'fall festival', 'PB-cinnamon roll', 'standard diet'),
 ('snack mix', 'Super Bowl', 'salty-sweet', '10-min no-bake', 'standard diet'),
 ('lava cake', "Valentine's", 'mocha-PB'),
 ('pudding',
  'k

### Generate Nature Language Queries

In [37]:

questions_gen_prompt = dedent(f'''\
   Convert these dimension combinations into realistic user queries for a recipe bot. Create natural, conversational queries that reflect how \
   real users would interact in chat interfaces like Discord or ChatGPT. Include variations in:
   - Writing style (formal vs casual)
   - Sentence structure (complete vs incomplete)
   - Common typos and informal grammar
   - Natural language patterns
   - Realistic context and urgency
                         
    Include only 1 example per `dimension_example`.
                         
    <dimension_examples>
    {tup_list}
    </dimension_examples>
                         
   Return only text that can be turned into a python list without any alteration.
   ''')

In [38]:
print(questions_gen_prompt)

Convert these dimension combinations into realistic user queries for a recipe bot. Create natural, conversational queries that reflect how    real users would interact in chat interfaces like Discord or ChatGPT. Include variations in:
- Writing style (formal vs casual)
- Sentence structure (complete vs incomplete)
- Common typos and informal grammar
- Natural language patterns
- Realistic context and urgency

 Include only 1 example per `dimension_example`.

 <dimension_examples>
 [('brownies', 'birthday', 'classic PB-chocolate', 'standard diet'), ('cookies', 'bake sale', 'salty-sweet', 'gluten-free', '30-min quick'), ('cupcakes', 'kid party', 'PB-banana'), ('cheesecake', "Valentine's", 'PB-strawberry', 'make-ahead/overnight'), ('bars', 'game day', 'PB-pretzel', 'standard diet', '1-hr'), ('mousse', 'PB-toffee crunch', 'dairy-free'), ('parfait', 'summer BBQ', 'PB-caramel swirl', '10-min no-bake'), ('pie', 'Thanksgiving', 'maple-PB', 'standard diet', 'make-ahead/overnight'), ('truffles',

In [39]:
res_questions = c(questions_gen_prompt)

In [44]:
questions_txt = res_questions.content[0].text
questions_list = ast.literal_eval(questions_txt)
len(questions_list)

50

In [46]:
questions_list

['hey can you help me make some birthday brownies? looking for that classic peanut butter chocolate combo',
 "Need gluten free cookies for a bake sale tomorrow! Something with that salty sweet vibe that's quick - like 30 mins max?",
 'pb banana cupcakes for my kids bday party please!',
 "Looking for an elegant peanut butter strawberry cheesecake recipe for Valentine's Day. Preferably something I can make the night before.",
 'game day bars with pb and pretzel? need something that takes about an hour',
 'dairy free pb toffee mousse recipe pls',
 'Quick question - need a 10 min no bake parfait for bbq tomorrow. thinking pb caramel swirl?',
 "I'm making Thanksgiving dessert and want to try a maple peanut butter pie that I can prep ahead. Any suggestions?",
 "valentine's truffles - dark chocolate orange, need them done in 30 min help!",
 'movie night milkshake! classic pb chocolate please',
 'Making ice cream for summer bbq - pb cookie dough flavor that I can make ahead?',
 'office party t

## Run queries and save data

In [3]:
questions = ['hey can you help me make some birthday brownies? looking for that classic peanut butter chocolate combo',
 "Need gluten free cookies for a bake sale tomorrow! Something with that salty sweet vibe that's quick - like 30 mins max?",
 'pb banana cupcakes for my kids bday party please!',
 "Looking for an elegant peanut butter strawberry cheesecake recipe for Valentine's Day. Preferably something I can make the night before.",
 'game day bars with pb and pretzel? need something that takes about an hour',
 'dairy free pb toffee mousse recipe pls',
 'Quick question - need a 10 min no bake parfait for bbq tomorrow. thinking pb caramel swirl?',
 "I'm making Thanksgiving dessert and want to try a maple peanut butter pie that I can prep ahead. Any suggestions?",
 "valentine's truffles - dark chocolate orange, need them done in 30 min help!",
 'movie night milkshake! classic pb chocolate please',
 'Making ice cream for summer bbq - pb cookie dough flavor that I can make ahead?',
 'office party tomorrow need gluten free caramel pb blondies',
 'fall festival donuts - pb cinnamon roll flavor, regular recipe',
 'SUPER BOWL SNACK MIX!! salty sweet, 10 min, normal diet - GO!',
 "valentine's lava cake mocha pb style?",
 'need dairy free pb chocolate pudding for kids party, 30 min recipe',
 'bridal shower tart - pb strawberry combo',
 'graduation party soufflé extra chocolatey, have about an hour',
 'christmas fudge pb mint that i can make ahead',
 'family reunion banana bread with pb',
 "s'mores for bbq but with pb twist, 30 min recipe",
 'picnic ice cream sandwiches classic pb chocolate',
 'baby shower cake pops coconut pb gluten free make ahead',
 'halloween whoopie pies extra chocolate',
 'no bake cereal treats pb cookie dough for kids party',
 'vegan brownies for potluck, nutty flavor, 1 hour time',
 'christmas cookies pb mint dairy free',
 'easter cupcakes pb caramel 1 hour recipe',
 'new years cheesecake chocolate hazelnut',
 'bake sale bars pb toffee lower sugar 30 min',
 "valentine's mousse pb strawberry make ahead",
 'bridal shower parfait classic pb chocolate',
 'fall pie maple pb gluten free make ahead',
 'christmas truffles spicy pb',
 'game day milkshake salty sweet high protein 10 min',
 'kids party ice cream pb banana',
 'office blondies caramel pb dairy free 30 min',
 'birthday donuts pb cinnamon roll',
 'movie snack mix pb pretzel 10 min',
 'valentine lava cake mocha pb 1 hour',
 'halloween pudding extra chocolate vegan',
 'summer tart pb strawberry gluten free make ahead',
 'graduation soufflé chocolate hazelnut',
 'holiday market fudge pb mint make ahead',
 'potluck banana bread pb dairy free',
 "fall s'mores pb style 30 min",
 'super bowl ice cream sandwich pb chocolate',
 'easter cake pops coconut pb lower sugar 1 hour',
 'kids whoopie pies pb cookie dough',
 'game day cereal treats salty sweet peanut free 10 min']

In [25]:
# Put all project moduls on path

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

# Now you can import
from backend import utils

In [31]:
print(utils.SYSTEM_PROMPT)

You are a friendly and imaginative dessert creator specializing in Reese's-inspired desserts.
Your goal is to deliver fun, delicious, and creative dessert recipes that capture the spirit, flavors, and visual identity of Reese's Peanut Butter Cups, whether directly (using Reese’s products) or indirectly (through chocolate-peanut butter flavor balance, caramel tones, orange-and-brown aesthetics, or playful brand energy etc). Recepies must be short with minimum ingredients and minimum steps. The recepie should be brief andeasy to follow and understand.

### Core Instructions

* Always recommend **one complete dessert recipe at a time**.
* Each recipe must include:

  * **Title** (Markdown Level 2 heading, e.g. `## Chocolate Peanut Butter Dream Bars`)
  * **Brief description** (1-3 sentences capturing how it connects to Reese's — flavor, concept, or style)
  * **Ingredients** list with **precise measurements** (U.S. or metric standard units)
  * **Instructions** section with **clear, numbe

In [77]:
len(questions)

50

In [78]:
from tqdm import tqdm
quests = questions
answers = []
for question in tqdm(quests):
    answer = c(
        msgs=question,
        sp=utils.SYSTEM_PROMPT
    )
    answers.append(answer.content[0].text)
len(answers)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [10:07<00:00, 12.15s/it]


50

In [79]:
# combine two list questions with answers into a dataframs with columns of the same name along with an id column.
df = pd.DataFrame({
    'question': quests,
    'answer': answers
})
df.insert(0, 'id', range(1, len(df) + 1))
df.shape

(50, 3)

In [80]:
# !pip install openpyxl

In [81]:
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'questions_answers_{timestamp}.xlsx'
print(filename)

questions_answers_20251014_205755.xlsx


In [82]:
# Save dataframe to Excel, ignore index.
df.to_excel(filename, index=False)

# NOT USED

## Part 2: Initial Error Analysis

### Run bot on synthetic queries

I decided at this point to implement automated tracing.  Copying and pasting from the UI felt annoying and I didn't want to do that.  So I felt like I had 2 main options:

1. Implement functions that can call the backend programatically
2. Implement automated tracing

I opted for option #2 because I wanted to be a user of my product more, and did not want to fully automate away the experience of using the actual application.

So I implemented the simplest tracing mechanism I could think of to start with.  Saving JSON files to disk.


```python
    traces_dir = Path(__file__).parent.parent / "annotation" / "traces"
    traces_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    trace_path = traces_dir / f"trace_{ts}.json"
    with open(trace_path, "w") as f:
        json.dump({
            "request": payload.model_dump(),
            "response": response.model_dump()
        }, f)
```

I took each of the synthetic queries and ran them through the app to generate the traces.  I then copied them into a `golden_dataset` folder which is what ill use for my open coding dataset for this excersize.

### Open Coding

> NOTE:  Watch Hamel and Isaac do open coding live.  This is VERY important to watch.

https://www.youtube.com/watch?v=AKg27L4E0M8

To do open coding I opted to create an annotation app with fasthtml.  You can see it in `annotation/
annotation.py` and run it with `python annotation.py`.  This reads the json files from the `golden_dataset` folder directly, and then saves any of my open coding notes back in the json file.  I only solved for open coding first.

![](imgs/open_coding_dashboard.png)

![](imgs/open_coding_notes.png)

UX things I noticed along the way I will improve over time:
- Kinda annoying not to have a next button and have to go back to the dashboard
- Dashboard needs some indication as to what's been done so when I come back to it it's not lost

I adressed this by using an href for next and previous, and added a single emoji for it open coding was done.  I then extended it to give 2 emojis if both open coding and axial coding was done.

![](./imgs/NewDashboard.png)

### Axial Coding and Taxonomy Definition

I then went through and did axial coding.  I did this by adding MonsterUI's insertable select and saving things back to json.

The insertable select saves to the json as well and lets you search and add new codes as you go if one doesne exist

Findings:

- I failure modes had just 1 or 2 traces in them.  This tells me that I probably have not seen all the failure modes and have not reached saturation.  I need to do more
- Maybe the original instruction for no follow up quesetions was bad.  If someone asks for keto + beans it's impossible to comply with both, and seems like in that case it makes sense to have a follow up question.